# Encoding Categorical Features

## Definition
ML models require **numeric input**. Categorical features (text labels) must be encoded as numbers.

### Encoding Strategies
| Method | When to Use | Pros | Cons |
|--------|-------------|------|------|
| **Label Encoding** | Ordinal data (S<M<L<XL) | Simple, preserves order | Implies magnitude between labels |
| **One-Hot Encoding** | Nominal data (no order) | No false ordinal relationship | High dimensionality with many cats |
| **Binary Encoding** | High cardinality | Less sparse than OHE | Less interpretable |
| **Target Encoding** | High cardinality + sufficient data | Captures signal | Risk of data leakage! |
| **Frequency Encoding** | High cardinality | Simple, no leakage | Loses category info |
| **Ordinal Encoding** | Ordinal categories | Correct for ordered data | — |

### Data Leakage Warning
> Target encoding MUST be computed only on training data and then applied to test data.
> Computing on full dataset leaks target information.

In [ ]:
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
n = 200
df = pd.DataFrame({
    'size':   rng.choice(['S', 'M', 'L', 'XL'], n),
    'color':  rng.choice(['Red', 'Blue', 'Green', 'Yellow', 'Purple'], n),
    'city':   rng.choice(['NYC', 'LA', 'Chicago', 'Houston', 'Phoenix',
                           'San Antonio', 'San Diego', 'Dallas'], n),
    'price':  rng.uniform(10, 200, n)
})
print(df.head(5))

In [ ]:
# 1. Label Encoding (ordinal)
size_order = {'S': 0, 'M': 1, 'L': 2, 'XL': 3}
df['size_encoded'] = df['size'].map(size_order)
print('Label encoded size:')
print(df[['size', 'size_encoded']].drop_duplicates().sort_values('size_encoded'))

In [ ]:
# 2. One-Hot Encoding with pd.get_dummies()
# For low cardinality features (< 10-15 unique values)
color_dummies = pd.get_dummies(df['color'], prefix='color')
df_ohe = pd.concat([df, color_dummies], axis=1)
print('One-hot encoded color:')
print(df_ohe[['color'] + list(color_dummies.columns)].head(6))

In [ ]:
# get_dummies options
# drop_first=True: drop one column to avoid multicollinearity (for linear models)
color_dummies_drop = pd.get_dummies(df['color'], prefix='color', drop_first=True)
print('OHE with drop_first:')
print(color_dummies_drop.head(5))
print('Shape:', color_dummies_drop.shape)

In [ ]:
# 3. Frequency Encoding (replace category with its frequency)
freq_map = df['city'].value_counts(normalize=True)
df['city_freq'] = df['city'].map(freq_map)
print('Frequency encoding:')
print(df[['city', 'city_freq']].drop_duplicates().sort_values('city_freq', ascending=False))

In [ ]:
# 4. Target Encoding (CAREFULLY — compute only on train split!)
rng2 = np.random.default_rng(42)
df['target'] = (df['price'] > df['price'].median()).astype(int)

# Simulate train/test split
train_mask = rng2.random(n) < 0.8
df_train = df[train_mask].copy()
df_test  = df[~train_mask].copy()

# Compute target encoding on TRAIN ONLY
target_enc = df_train.groupby('city')['target'].mean()
print('City target encoding (from train):')
print(target_enc.round(3))

# Apply to both train and test
df_train['city_target_enc'] = df_train['city'].map(target_enc)
df_test['city_target_enc']  = df_test['city'].map(target_enc)

# Fill NaN for cities in test not in train
df_test['city_target_enc'] = df_test['city_target_enc'].fillna(df_train['target'].mean())
print('Target encoded (no leakage):')
print(df_test[['city', 'city_target_enc']].head(5))

In [ ]:
# 5. Ordinal Encoding with sklearn
from sklearn.preprocessing import OrdinalEncoder

oe = OrdinalEncoder(categories=[['S', 'M', 'L', 'XL']])
df['size_ordinal'] = oe.fit_transform(df[['size']])
print('Sklearn ordinal encoding:')
print(df[['size', 'size_ordinal']].drop_duplicates().sort_values('size_ordinal'))

In [ ]:
# 6. One-hot with sklearn OneHotEncoder (handles unseen categories in test!)
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
color_arr = ohe.fit_transform(df[['color']])
color_df = pd.DataFrame(color_arr, columns=ohe.get_feature_names_out(['color']))
print('Sklearn OHE shape:', color_df.shape)
print(color_df.head(3))